In [ ]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 加载数据
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/smoker_status_Bio_Signals/train.csv'
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/smoker_status_Bio_Signals/test.csv'

train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

# 查看数据的基本信息
print(train_df.head())
print(test_df.head())


       id  age  height  weight  waist  ...  AST  ALT  Gtp  dental_caries  smoking
0   60700   40     150      50   80.0  ...   14   11    9              0        0
1   44065   65     150      50   69.0  ...   17   24   25              0        0
2   39538   55     155      55   80.0  ...   19   15   16              0        0
3  105427   55     160      60   83.0  ...   14   13   26              0        0
4  148669   30     180      90   95.0  ...   25   30   21              0        0

[5 rows x 24 columns]
       id  age  height  weight  waist  ...  AST  ALT  Gtp  dental_caries  smoking
0  145654   35     175      80   84.0  ...   31   22   32              0        1
1   49118   35     185      80   88.0  ...   22   22   17              0        1
2   21769   20     160      60   76.0  ...   24   32   41              1        1
3  108299   60     155      60   87.8  ...   21   16   14              0        0
4  117130   60     165      70   85.0  ...   27   40   61              0   

In [ ]:

# 数据预处理
# 检查缺失值
print("训练集缺失值情况:\n", train_df.isnull().sum())
print("测试集缺失值情况:\n", test_df.isnull().sum())

# 编码分类变量
train_df['dental_caries'] = train_df['dental_caries'].astype('category')
test_df['dental_caries'] = test_df['dental_caries'].astype('category')

# 对分类变量进行one-hot编码
train_df = pd.get_dummies(train_df, columns=['dental_caries'])
test_df = pd.get_dummies(test_df, columns=['dental_caries'])

# 检查数据类型
print(train_df.dtypes)
print(test_df.dtypes)


Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

        0
age                    0
height                 0
weight                 0
waist                  0
eyesight_left          0
eyesight_right         0
hearing_left           0
hearing_right          0
systolic               0
relaxation             0
fasting_blood_sugar    0
Cholesterol            0
triglyceride           0
HDL                    0
LDL                    0
hemoglobin             0
Urine_protein          0
serum_creatinine       0
AST                    0
ALT                    0
Gtp                    0
dental_caries          0
smoking                0
dtype: int64
测试集缺失值情况:
 id                     0
age                    0
height                 0
weight                 0
waist                  0
eyesight_left          0
eyesight_right         0
hearing_left           0
hearing_right          0


In [ ]:


# 准备数据
X_train = train_df.drop(columns=['smoking', 'id'])
y_train = train_df['smoking']

X_test = test_df.drop(columns=['smoking', 'id'])
y_test = test_df['smoking']

# 数据标准化
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 建立XGBoost模型
params = {
    'objective': 'binary:logistic',
    'booster': 'dart',
    'eta': 0.1,
    'eval_metric': 'auc',
    'seed': 42
}

dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
dtest = xgb.DMatrix(X_test_scaled, label=y_test)

# 训练模型
model = xgb.train(params, dtrain, num_boost_round=100)

# 评估模型
y_pred_proba = model.predict(dtest)
auc_roc = roc_auc_score(y_test, y_pred_proba)

print(f"AUC-ROC: {auc_roc:.4f}")



AUC-ROC: 0.8617
